# ISE 5406 — Final Exam (Spring 2026)

This notebook is your **partial template** for the take-home Final. The setup cells, helper functions, and CVXPY baselines are filled in; cells marked **TODO** are for you to fill in.

**How to use it.**
1. Run the setup cells in §0 — they import everything and define the helpers you'll reuse.
2. Each problem has: a brief restatement, the data loader, sometimes a CVXPY baseline, and one or more TODO cells where you write the algorithmic content.
3. The TODO cells contain function signatures, docstrings with hints, and `raise NotImplementedError` placeholders. Replace each `raise` with your implementation.
4. Markdown TODO sections ask for 1–2 paragraph discussions or theory derivations. Replace the `*your work here*` text with your write-up.

**Helpers you can use freely** (defined in §0):
`standardize_features`, `train_test_split`, `plot_loss_curves`, `armijo_backtrack`, `check_my_gradient`, `load_case9`, `unpack_state`, `pack_state`, `make_bounds`, `flat_start`, `random_start`, `power_residuals`, `gen_cost`, `solve_acopf_slsqp`, `expdecay_model`, `expdecay_residuals`.

**AI policy reminder.** Use AI for code/LaTeX, NOT for proofs or algorithm derivations. Disclose all AI use in your final write-up.


## 0. Setup

Imports + helper functions. Each helper has a brief math/intuition cell before its code so you understand what it does.


### 0.0 Standard imports

NumPy + SciPy for numerics, matplotlib for plotting, CVXPY for convex baselines, scipy.optimize for L-BFGS / SLSQP / trust-NCG. Locking the random seed at the top makes everything reproducible.


In [ ]:
import os, time, warnings
import numpy as np
import scipy.linalg as sla
from scipy.optimize import minimize, check_grad
import matplotlib.pyplot as plt
import cvxpy as cp
warnings.filterwarnings('ignore')

DATA_DIR = './data'
rng = np.random.default_rng(2026)
print('numpy', np.__version__, '|', 'cvxpy', cp.__version__)


### 0.1 Python tricks worth knowing

A few idioms that recur on this exam.

| Trick | Use it for | One-liner |
|---|---|---|
| `np.logaddexp(0, -z)` | Stable $\log(1+e^{-z})$ in logistic loss (P2) | overflows are gone |
| `sla.cho_factor` + `sla.cho_solve` | Cache a Cholesky factor; solve many RHSes (P4 ALM, P4 ridge) | $O(n^2)$ per solve, not $O(n^3)$ |
| `minimize(..., callback=cb)` | Record convergence history (P2 L-BFGS, P3 SLSQP) | closure that appends to a list |
| `check_grad(f, grad, x)` | Verify analytic gradient before trusting it | should return $\sim 10^{-6}$ |
| `np.random.default_rng(seed)` | Reproducibility | one RNG passed around |
| Broadcasting | Standardize without loops | `(X - X.mean(0)) / X.std(0)` |
| `fig, axes = plt.subplots(...)` | Multi-panel plots | preferred over global pyplot |


### 0.2 `standardize_features(X)`

**What it does.** Replaces each column $X_{:,j}$ with $(X_{:,j} - \bar{X}_j) / s_j$ where $\bar{X}_j$ is the column mean and $s_j$ the column standard deviation. Result: every column has zero mean and unit variance.

**Why you need this on this exam.** Problem 4 option (iv) has features whose standard deviations span 6 orders of magnitude. Adam's per-coordinate adaptive rate $\eta / \sqrt{\hat v}$ helps but cannot fully compensate for that scale spread — the canonical fix is to standardize first, then optimize.

**Implementation note.** Returns the (mean, std) so you can apply the *same* transform to test data: `X_test_std = (X_test - mu) / sd`. Never re-fit standardization on the test set.


In [ ]:
def standardize_features(X):
    """Subtract column mean, divide by column std. Returns (X_std, mean, std)."""
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0           # avoid divide-by-zero on constant columns
    return (X - mu) / sd, mu, sd


### 0.3 `train_test_split(A, y, test_frac, seed)`

**What it does.** Random shuffle of the row indices, then split into a training fraction $1 - p$ and a test fraction $p$. Reproducible — same `seed` ⇒ same split.

**Why standard ML hygiene.** Train and test sets must come from the *same distribution* but be *disjoint subsets*. The shuffle is what guarantees this when the data file might be sorted or grouped (e.g., all positives first, then all negatives). Reproducible splits make grading possible — your test accuracy with `seed=0` should match the reference solution.


In [ ]:
def train_test_split(A, y, test_frac=0.2, seed=0):
    """Random shuffle + split. Returns (A_tr, y_tr, A_te, y_te)."""
    _r = np.random.default_rng(seed)
    perm = _r.permutation(len(y))
    cut = int((1 - test_frac) * len(y))
    return A[perm[:cut]], y[perm[:cut]], A[perm[cut:]], y[perm[cut:]]


### 0.4 `plot_loss_curves(histories, labels, ...)`

**What it does.** Overlays multiple convergence histories on a log-scale y-axis.

**Why log-y.** Convergence rates are most clearly visualized on a log scale: linear convergence ($\rho^k$) becomes a straight line with slope $\log \rho$, and the difference between $O(1/k)$ and $O(1/k^2)$ is visually obvious. On a linear axis, the curves all crash to zero and you can't tell them apart.


In [ ]:
def plot_loss_curves(histories, labels, title='Convergence',
                     xlabel='iteration', ylabel='loss', log_y=True, figsize=(7, 4)):
    """Overlay loss-vs-iteration curves on a log-y axis."""
    fig, ax = plt.subplots(figsize=figsize)
    for h, lab in zip(histories, labels):
        h = np.asarray(h, dtype=float); h = h[np.isfinite(h)]
        (ax.semilogy if log_y else ax.plot)(h, label=lab, marker='.', markersize=3, linewidth=1)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3); ax.legend()
    return ax


### 0.5 `armijo_backtrack(f, x, dx, f0, g0, ...)`

**What it does.** Find a step size $s \in (0, 1]$ such that the **Armijo sufficient-decrease condition** holds:

$$f(x + s\,\Delta x) \;\le\; f(x) + \alpha\, s\, g^\top \Delta x.$$

Start with $s = 1$; if the condition fails, halve $s$ (multiply by $\beta = 0.5$) and try again.

**Why this works.** The right-hand side is a strictly-better-than-zero linear lower bound on the predicted decrease. Choosing $\alpha = 0.25$ means we accept any step that achieves at least 25% of the predicted decrease. Backtracking is guaranteed to terminate as long as $\Delta x$ is a *descent direction* ($g^\top \Delta x < 0$): for sufficiently small $s$ the actual decrease equals the predicted decrease (Taylor expansion), so the condition eventually holds.

**Where you'll use it.** Inside the IPM Newton step (Problem 1) and the Newton/Gauss-Newton inner loops (Problem 4 option ii).


In [ ]:
def armijo_backtrack(f, x, dx, f0, g0, alpha=0.25, beta=0.5, max_iter=50):
    """Armijo line search: find s with f(x + s*dx) <= f0 + alpha*s*(g0@dx). Returns (s, x_new)."""
    s = 1.0
    g_dx = float(g0 @ dx)
    for _ in range(max_iter):
        if f(x + s * dx) <= f0 + alpha * s * g_dx:
            return s, x + s * dx
        s *= beta
    return s, x + s * dx


### 0.6 `check_my_gradient(f_and_grad, x, eps)`

**What it does.** Computes $\|\nabla f_{\text{analytic}}(x) - \nabla f_{\text{finite-diff}}(x)\|_2$ where the finite-difference gradient is

$$[\nabla f_{\text{fd}}]_i \;\approx\; \frac{f(x + \varepsilon e_i) - f(x - \varepsilon e_i)}{2\varepsilon}.$$

**Why use it.** Hand-derived gradients are an extremely common source of bugs (signs flipped, factors of $1/N$ missing, indexing off by one). Optimizers will often *appear* to work with a buggy gradient — you'll see decreasing loss but converge to the wrong point, or convergence will be much slower than expected.

**Tolerance.** A correct gradient at a smooth point should give error $\sim 10^{-6}$ (the central-difference truncation error for $\varepsilon = 10^{-6}$). Errors above $10^{-3}$ almost certainly indicate a bug.


In [ ]:
def check_my_gradient(f_and_grad, x, eps=1e-6):
    """Wrap scipy.optimize.check_grad for f_and_grad(x) -> (loss, grad)."""
    return check_grad(lambda z: f_and_grad(z)[0],
                      lambda z: f_and_grad(z)[1], x, epsilon=eps)


### 0.7 AC OPF state-vector convention

The AC OPF problem (Problem 3) has $4 n_b$ decision variables: voltage magnitudes $|V| \in \mathbb{R}^{n_b}$, voltage angles $\theta \in \mathbb{R}^{n_b}$, real-power generation $P_g \in \mathbb{R}^{n_b}$, and reactive-power generation $Q_g \in \mathbb{R}^{n_b}$.

For scipy.optimize, we pack these into a single flat vector $x \in \mathbb{R}^{4 n_b}$ in the order $x = [|V|;\, \theta;\, P_g;\, Q_g]$. The next few helpers convert between the flat vector and the four named arrays.


### 0.8 `load_case9(path)`

**What it does.** Loads `data/case9.npz` and returns a dict with all the network constants:

| Key | Shape | Meaning |
|---|---|---|
| `G`, `B` | $9 \times 9$ | Per-unit admittance matrix $Y = G + jB$ |
| `P_d`, `Q_d` | $(9,)$ | Real / reactive load at each bus (per-unit) |
| `P_g_min/max`, `Q_g_min/max` | $(9,)$ | Generator bounds (zero at non-generator buses) |
| `c2`, `c1`, `c0` | $(9,)$ | Quadratic generator cost: $c_{2,i} P_{g,i}^2 + c_{1,i} P_{g,i} + c_{0,i}$ |
| `V_min`, `V_max` | $(9,)$ | Voltage magnitude bounds |
| `nb` (added) | int | Number of buses (9) |


In [ ]:
def load_case9(path=os.path.join(DATA_DIR, 'case9.npz')):
    """Load IEEE 9-bus AC OPF data into a dict."""
    d = np.load(path)
    net = {k: d[k] for k in d.files}
    net['nb'] = net['G'].shape[0]
    return net


### 0.9 `unpack_state(x, nb)` and `pack_state(V, theta, Pg, Qg)`

**What they do.** Given the flat vector $x \in \mathbb{R}^{4 n_b}$, `unpack_state` returns the four named arrays $(|V|, \theta, P_g, Q_g)$. `pack_state` is the inverse — concatenate the four arrays into a flat vector for scipy.


In [ ]:
def unpack_state(x, nb):
    """x -> (V, theta, Pg, Qg)."""
    return x[:nb], x[nb:2*nb], x[2*nb:3*nb], x[3*nb:4*nb]


def pack_state(V, theta, Pg, Qg):
    """Concatenate (V, theta, Pg, Qg) into a single x."""
    return np.concatenate([V, theta, Pg, Qg])


### 0.10 `make_bounds(net)`, `flat_start(net)`, `random_start(net, rng)`

**`make_bounds`.** Constructs the list of $(\text{low}, \text{high})$ tuples that scipy's SLSQP needs:
- Voltages bounded by $[V_i^{\min}, V_i^{\max}]$
- Angles bounded by $[-\pi, \pi]$ (only $\theta_1 = 0$ is enforced as an equality)
- Generator powers bounded by $[P_g^{\min}, P_g^{\max}]$ (and similarly for $Q_g$)

**`flat_start`.** The canonical AC OPF initial guess: $|V|_i = 1$ pu (nominal voltage), $\theta_i = 0$ (no power flow), generators at the midpoint of their range. Power-system-engineering jargon: "flat voltage profile."

**`random_start`.** For the multi-start study in part (c). Voltages uniform in a small range around 1 pu, angles uniform in a small symmetric range, generators randomized within their bounds. The slack-bus angle $\theta_1$ is pinned to 0.


In [ ]:
def make_bounds(net):
    """List of (low, high) tuples for [V; theta; Pg; Qg]."""
    nb = net['nb']
    bnds = []
    for i in range(nb): bnds.append((float(net['V_min'][i]), float(net['V_max'][i])))
    for i in range(nb): bnds.append((-np.pi, np.pi))
    for i in range(nb): bnds.append((float(net['P_g_min'][i]), float(net['P_g_max'][i])))
    for i in range(nb): bnds.append((float(net['Q_g_min'][i]), float(net['Q_g_max'][i])))
    return bnds


def flat_start(net):
    """|V|=1, theta=0, generators at midpoint of their range."""
    nb = net['nb']
    return pack_state(np.ones(nb), np.zeros(nb),
                      0.5 * (net['P_g_min'] + net['P_g_max']),
                      0.5 * (net['Q_g_min'] + net['Q_g_max']))


def random_start(net, rng_, V_lo=0.95, V_hi=1.05, theta_range=np.pi/12):
    """Random initial guess for multi-start. Slack bus angle pinned to 0."""
    nb = net['nb']
    V = rng_.uniform(V_lo, V_hi, nb)
    th = rng_.uniform(-theta_range, theta_range, nb); th[0] = 0.0
    Pg = rng_.uniform(net['P_g_min'], np.maximum(net['P_g_max'], net['P_g_min'] + 1e-6))
    Qg = rng_.uniform(net['Q_g_min'], np.maximum(net['Q_g_max'], net['Q_g_min'] + 1e-6))
    return pack_state(V, th, Pg, Qg)


### 0.11 `power_residuals(x, net, alpha)`

**What it computes.** The equality-constraint residual vector for the AC OPF problem.

For each bus $i$, the per-unit real and reactive power **injected** by the network is

$$P_{\text{inj},i} = |V_i| \sum_k |V_k|\bigl(G_{ik}\cos(\theta_i - \theta_k) + B_{ik}\sin(\theta_i - \theta_k)\bigr),$$
$$Q_{\text{inj},i} = |V_i| \sum_k |V_k|\bigl(G_{ik}\sin(\theta_i - \theta_k) - B_{ik}\cos(\theta_i - \theta_k)\bigr).$$

The power-balance equality at bus $i$ is

$$P_{g,i} - P_{d,i} - P_{\text{inj},i} = 0, \qquad Q_{g,i} - Q_{d,i} - Q_{\text{inj},i} = 0.$$

This function returns the concatenated residual vector $[P_g - P_d - P_{\text{inj}};\; Q_g - Q_d - Q_{\text{inj}};\; \theta_1]$ — the trailing $\theta_1$ pins the slack bus angle to zero. SLSQP drives this whole vector to zero.

**The `alpha` parameter** scales the loads $P_d, Q_d$ — used in part (d) for the stress test.

**The double for-loop** is a literal transcription of the formulas above. It runs in $O(n_b^2)$ which is fine for $n_b = 9$ but inefficient for larger systems. (You could vectorize it; we keep it in long form for clarity.)


In [ ]:
def power_residuals(x, net, alpha=1.0):
    """Concatenated [P_balance; Q_balance; theta_slack] equality residuals."""
    nb = net['nb']
    V, th, Pg, Qg = unpack_state(x, nb)
    G, B_ = net['G'], net['B']
    Pd, Qd = alpha * net['P_d'], alpha * net['Q_d']
    P_inj = np.zeros(nb); Q_inj = np.zeros(nb)
    for i in range(nb):
        for k in range(nb):
            cd = np.cos(th[i] - th[k]); sd = np.sin(th[i] - th[k])
            P_inj[i] += V[i] * V[k] * (G[i, k] * cd + B_[i, k] * sd)
            Q_inj[i] += V[i] * V[k] * (G[i, k] * sd - B_[i, k] * cd)
    return np.concatenate([Pg - Pd - P_inj, Qg - Qd - Q_inj, [th[0]]])


### 0.12 `gen_cost(x, net)` and `solve_acopf_slsqp(x0, net, alpha)`

**`gen_cost`.** The objective: $\sum_i (c_{2,i} P_{g,i}^2 + c_{1,i} P_{g,i} + c_{0,i})$. At non-generator buses the cost coefficients are zero so those terms drop.

**`solve_acopf_slsqp`.** Wraps `scipy.optimize.minimize(method='SLSQP')` with the bounds and equality constraints from above. SLSQP is an SQP-style local NLP solver — the closest standard fallback when IPOPT isn't installed. For Problem 3 part (a), the *intended* solver is Pyomo + IPOPT (which you'll write the model for); this SLSQP path is the in-notebook backup.


In [ ]:
def gen_cost(x, net, alpha=1.0):
    """Quadratic generator cost."""
    nb = net['nb']
    _, _, Pg, _ = unpack_state(x, nb)
    return float(np.sum(net['c2'] * Pg**2 + net['c1'] * Pg + net['c0']))


def solve_acopf_slsqp(x0, net, alpha=1.0):
    """SLSQP solve. Returns the scipy OptimizeResult."""
    return minimize(
        gen_cost, x0, args=(net, alpha), method='SLSQP',
        bounds=make_bounds(net),
        constraints=[{'type': 'eq', 'fun': lambda x: power_residuals(x, net, alpha)}],
        options=dict(maxiter=300, ftol=1e-7),
    )


### 0.13 `expdecay_model(t, params)` and `expdecay_residuals(t, y, params)`

**What it computes.** The exponential-decay model $y(t) = a + b\,e^{-c t}$ for parameters $\theta = (a, b, c) \in \mathbb{R}^3$, plus its residual vector $r_i(\theta) = y_i - (a + b e^{-c t_i})$.

**What's *not* given (you derive).** The Jacobian $J = \partial r / \partial \theta$ — you'll need this for both Gauss-Newton and full Newton in Problem 4 option (ii). Hint: it's $J_i = (-1,\ -e^{-c t_i},\ b\, t_i\, e^{-c t_i})$.

**Why split model from residuals.** Lets you use the model to plot fits, compute predictions on new $t$, etc., without having to redo the residual subtraction. Standard NLS API split.


In [ ]:
def expdecay_model(t, params):
    """Exponential-decay model y = a + b * exp(-c * t)."""
    a, b, c = params
    return a + b * np.exp(-c * t)


def expdecay_residuals(t, y, params):
    """r_i = y_i - model(t_i, params)."""
    return y - expdecay_model(t, params)


print('All helpers loaded.')


---

## 1. Problem 1 — Markowitz portfolio QP [20 pts]

Solve the long-only fully-invested mean-variance problem
$$\min_w\ \tfrac{\gamma}{2} w^\top \Sigma w - \boldsymbol\mu^\top w \quad\text{s.t.}\ \mathbf{1}^\top w = 1,\ w \ge 0,$$
with $\gamma = 5$.

### 1.1 Load data and CVXPY baseline (filled in)


In [ ]:
d = np.load(os.path.join(DATA_DIR, 'portfolio30.npz'))
Sigma, mu_p = d['Sigma'], d['mu']
n_p = len(mu_p); GAMMA = 5.0
print(f'Loaded portfolio30: n = {n_p}, kappa(Sigma) = {np.linalg.eigvalsh(Sigma)[-1]/np.linalg.eigvalsh(Sigma)[0]:.0f}')

def f_obj(w, Sigma=Sigma, mu=mu_p, gamma=GAMMA):
    return float(0.5 * gamma * w @ Sigma @ w - mu @ w)

w_var = cp.Variable(n_p)
prob = cp.Problem(cp.Minimize(0.5 * GAMMA * cp.quad_form(w_var, cp.psd_wrap(Sigma)) - mu_p @ w_var),
                  [w_var >= 0, cp.sum(w_var) == 1])
t0 = time.time(); prob.solve(solver='CLARABEL'); t_cvx = time.time() - t0
f_cvx = prob.value
print(f'CVXPY: f = {f_cvx:.6f}, time = {t_cvx*1000:.1f} ms')


### 1.2 Simplex projection (you write — proved in P6)

You'll need this for NAPG below. The closed form (proved in Problem 6 part (b)–(d)):

$$P_\Delta(v) = \max(v - \theta^\star, 0)$$

with $\theta^\star = \frac{1}{\rho}(\sum_{j \le \rho} u_{(j)} - 1)$, $\rho = \max\{j : u_{(j)} - \frac{1}{j}(\sum_{i \le j} u_{(i)} - 1) > 0\}$, $u_{(1)} \ge u_{(2)} \ge \cdots$ the sorted entries of $v$.


In [ ]:
def project_simplex(v):
    """Closed-form projection onto {w >= 0, 1'w = 1}.

    Hints:
        u    = np.sort(v)[::-1]              # descending sort
        cssv = np.cumsum(u) - 1.0            # cumulative-sum-shifted
        rho  = largest j (1-indexed) such that u_(j) - (1/j)*(sum_{i<=j} u_(i) - 1) > 0
        theta = cssv[rho] / (rho + 1)        # using 0-indexed rho here
        return np.maximum(v - theta, 0.0)
    """
    raise NotImplementedError("Implement simplex projection here")


# Test: should match CVXPY to machine precision
v_test = np.array([-1.0, 0.5, 2.0, 0.1, -0.3, 1.5])
# w = project_simplex(v_test)
# print(f'closed form: {np.round(w, 4)}')
# print(f'sum(w) = {w.sum():.6f}, min(w) = {w.min():.6f}')


### 1.3 Log-barrier interior-point method (you write)

Centered objective: $L_t(w) = \tfrac{\gamma}{2} w^\top \Sigma w - \boldsymbol\mu^\top w - \tfrac{1}{t}\sum_i \log w_i$.

Newton step (preserve $\mathbf{1}^\top w = 1$): solve

$$\begin{bmatrix} H & A^\top \\ A & 0 \end{bmatrix}\begin{bmatrix}\Delta w \\ \nu^+\end{bmatrix} = -\begin{bmatrix} \nabla L_t \\ 0 \end{bmatrix}, \quad A = \mathbf{1}^\top.$$

Schur: $\nu^+ = -(A H^{-1} A^\top)^{-1} A H^{-1} \nabla L_t$, $\Delta w = -H^{-1}(\nabla L_t + A^\top \nu^+)$.

Use **fraction-to-boundary** then `armijo_backtrack` (from §0.5). Outer loop: $t \leftarrow 10\,t$ from $t_0 = 1$ until $n/t < 10^{-8}$.


In [ ]:
def solve_ipm(Sigma, mu, gamma, t0=1.0, mu_t=10.0, eps=1e-8,
              max_outer=30, max_inner=50):
    """Log-barrier IPM with equality-constrained Newton. Returns (w, time, outer, inner)."""
    n = len(mu); A = np.ones((1, n))
    w = np.full(n, 1.0/n); t = t0
    outer = 0; inner = 0; t_start = time.time()
    while True:
        outer += 1
        for k in range(max_inner):
            inner += 1
            # 1. Compute gradient and Hessian of L_t
            grad_t = ...  # TODO: gamma * Sigma @ w - mu - (1/t) / w
            H      = ...  # TODO: gamma * Sigma + (1/t) * diag(1/w^2)

            # 2. Schur-complement Newton solve
            #    Hinv_AT  = sla.solve(H, A.T, assume_a='pos')
            #    Hinv_g   = sla.solve(H, grad_t, assume_a='pos')
            #    nu_plus  = -sla.solve(A @ Hinv_AT, A @ Hinv_g)
            #    dw       = -(Hinv_g + Hinv_AT @ nu_plus)

            # 3. Newton-decrement stopping criterion: if -grad_t @ dw / 2 < 1e-10: break

            # 4. Fraction-to-boundary step + Armijo backtracking via armijo_backtrack(...)

            raise NotImplementedError("Implement IPM Newton step here")
        if n / t < eps or outer >= max_outer:
            break
        t *= mu_t
    return w, time.time() - t_start, outer, inner


# Test
# w_ipm, t_ipm, outer, inner = solve_ipm(Sigma, mu_p, GAMMA)
# print(f'IPM: f = {f_obj(w_ipm):.6f}, time = {t_ipm*1000:.1f} ms, outer = {outer}, inner = {inner}')


### 1.4 Nesterov-accelerated projected gradient (you write)

With $L = \gamma \lambda_{\max}(\Sigma)$, $\mu_{\rm sc} = \gamma \lambda_{\min}(\Sigma)$, $\kappa = L/\mu_{\rm sc}$, $\beta = (\sqrt\kappa - 1)/(\sqrt\kappa + 1)$:

Initialize $w_0 = (1/n)\mathbf{1}$, $y_0 = w_0$. Repeat:
1. $g_k = \nabla f(y_k) = \gamma \Sigma y_k - \boldsymbol\mu$
2. $w_{k+1} = P_\Delta(y_k - g_k / L)$
3. $y_{k+1} = w_{k+1} + \beta(w_{k+1} - w_k)$


In [ ]:
def solve_napg(Sigma, mu, gamma, max_iter=2000, tol=1e-10):
    """Nesterov-accelerated projected gradient on the simplex. Returns (w, history, time)."""
    n = len(mu)
    eigs = np.linalg.eigvalsh(Sigma)
    L = gamma * eigs[-1]
    mu_sc = gamma * eigs[0]
    # TODO: kappa, beta from L and mu_sc

    w = np.full(n, 1.0/n)
    y = w.copy()
    history = [f_obj(w, Sigma, mu, gamma)]
    t0 = time.time()
    for k in range(max_iter):
        # TODO: gradient at look-ahead y, projected step, momentum update
        raise NotImplementedError("Implement NAPG main loop here")
    return w, history, time.time() - t0


# Test
# w_n, hist_napg, t_n = solve_napg(Sigma, mu_p, GAMMA)
# print(f'NAPG: f = {hist_napg[-1]:.6f}, time = {t_n*1000:.2f} ms, iters = {len(hist_napg)}')


### 1.5 Algorithm-selection essay (you write — markdown)

*Write 1–2 paragraphs covering each of the following scenarios. Which method dominates and why?*

1. **Real-time rebalancing** with a new $\boldsymbol\mu$ each minute (warm-starting matters).
2. **One-off allocation** with high precision required.
3. **Sensitivity sweep** over 10,000 candidate $\Sigma$ matrices.

*your discussion here*


### 1.6 Efficient frontier (you write)

**What is the efficient frontier?** The locus of $(\sigma, \mathbb{E}[r])$ pairs $(\sqrt{w^\top \Sigma w},\ \boldsymbol\mu^\top w)$ as you sweep the risk-aversion parameter $\gamma$. Each $\gamma$ produces one optimal portfolio, hence one point on the frontier.

**Interpretation.** Small $\gamma$ → barely risk-averse → optimum collapses to the single highest-return asset (top-right corner of the frontier). Large $\gamma$ → very risk-averse → optimum approaches the global minimum-variance portfolio (bottom-left). The curve traced is a hyperbola-like Pareto front.

**Your task.** Sweep $\gamma$ over $\{0.5, 1, 2, 5, 10, 25, 100\}$. For each $\gamma$, solve the Markowitz problem (use any of CVXPY, your IPM, or your NAPG — CVXPY is easiest). Record $(\gamma,\ \sigma,\ \mathbb{E}[r])$ and the size of the active set (number of $w_i > 10^{-6}$). Plot $\sigma$ vs $\mathbb{E}[r]$ as a labeled line.


In [ ]:
# TODO: efficient frontier sweep + plot
#
# Outline:
#   gammas = [0.5, 1.0, 2.0, 5.0, 10.0, 25.0, 100.0]
#   For each gamma:
#       solve the Markowitz problem (CVXPY or your IPM/NAPG)
#       record (gamma, std, return, active_set_size)
#   Plot std vs return; annotate each point with its gamma.

raise NotImplementedError("Implement the efficient-frontier sweep here")


---

## 2. Problem 2 — Logistic regression on UCI WDBC [20 pts]

$$\min_{w, b}\ \frac{1}{N}\sum_i \log(1 + e^{-y_i(w^\top a_i + b)}) + \frac{\lambda}{2}\|w\|^2,\quad \lambda = 0.01.$$

### 2.1 Load + 80/20 split (filled in)


In [ ]:
d = np.load(os.path.join(DATA_DIR, 'wdbc.npz'))
A_lr, y_lr = d['A'], d['y']
A_tr, y_tr, A_te, y_te = train_test_split(A_lr, y_lr, test_frac=0.2, seed=0)
LAM_LR = 0.01
print(f'train {A_tr.shape}, test {A_te.shape}')


### 2.2 Gradient and verification (you write)

Derive the gradient analytically, then verify it with `check_my_gradient` (from §0.6).

Let $z_i = y_i(w^\top a_i + b)$ and $\sigma_i = 1/(1 + e^{z_i})$. Then $\nabla_w L = -\frac{1}{N}\sum_i \sigma_i y_i a_i + \lambda w$, $\partial L / \partial b = -\frac{1}{N}\sum_i \sigma_i y_i$. Use `np.logaddexp(0, -z)` for stable loss evaluation.


In [ ]:
def lr_loss_and_grad(theta, A, y, lam):
    """Logistic loss and gradient. theta = [w; b]."""
    N, p = A.shape
    w, b = theta[:p], theta[p]
    # TODO: z_i = y_i (a_i' w + b)
    # TODO: stable loss using np.logaddexp
    # TODO: sigma_i = 1 / (1 + exp(z_i))
    # TODO: grad_w, grad_b
    raise NotImplementedError("Implement logistic loss and gradient")


def lr_loss(theta, A, y, lam): return lr_loss_and_grad(theta, A, y, lam)[0]
def lr_acc(theta, A, y):
    p = A.shape[1]
    return float((np.where(A @ theta[:p] + theta[p] >= 0, 1, -1) == y).mean())


# Verify: should print ~1e-5 or smaller
# x0 = rng.standard_normal(A_tr.shape[1] + 1)
# err = check_my_gradient(lambda t: lr_loss_and_grad(t, A_tr, y_tr, LAM_LR), x0)
# print(f'Analytic gradient error: {err:.2e}')


### 2.3 Four methods

CVXPY baseline is filled in. **You write** L-BFGS, SGD, and your-choice fourth method.


In [ ]:
results = {}
wv = cp.Variable(A_tr.shape[1]); bv = cp.Variable()
logloss = cp.sum(cp.logistic(cp.multiply(-y_tr.astype(float), A_tr @ wv + bv))) / len(y_tr)
prob = cp.Problem(cp.Minimize(logloss + 0.5 * LAM_LR * cp.sum_squares(wv)))
t0 = time.time(); prob.solve(solver='CLARABEL'); t = time.time() - t0
results['cvxpy'] = dict(theta=np.concatenate([wv.value, [bv.value]]), time=t)
print(f'CVXPY: loss = {lr_loss(results["cvxpy"]["theta"], A_tr, y_tr, LAM_LR):.6f}, time = {t:.3f}s')


In [ ]:
# TODO: L-BFGS via scipy.optimize.minimize using your lr_loss_and_grad
# Use the callback closure pattern from §0.1 to record convergence history.

# history_lbfgs = []
# def cb(theta, A=A_tr, y=y_tr): history_lbfgs.append(lr_loss(theta, A, y, LAM_LR))
# t0 = time.time()
# res = minimize(lr_loss_and_grad, np.zeros(A_tr.shape[1] + 1),
#                args=(A_tr, y_tr, LAM_LR), jac=True, method='L-BFGS-B',
#                callback=cb, options=dict(maxiter=200, ftol=1e-10, gtol=1e-8))
# results['lbfgs'] = dict(theta=res.x, time=time.time()-t0, history=history_lbfgs)

raise NotImplementedError("Implement L-BFGS section here")


In [ ]:
# TODO: SGD from scratch.
# Pick and document: step-size schedule, batch size, # epochs.

# theta = np.zeros(A_tr.shape[1] + 1); history_sgd = []
# _r = np.random.default_rng(2026)
# for ep in range(?):                # YOUR CHOICE
#     perm = _r.permutation(len(y_tr))
#     for s in range(0, len(y_tr), ?):    # YOUR CHOICE batch size
#         idx = perm[s:s+?]
#         lr_rate = ?                # YOUR CHOICE step-size schedule
#         _, g = lr_loss_and_grad(theta, A_tr[idx], y_tr[idx], LAM_LR)
#         theta = theta - lr_rate * g
#     history_sgd.append(lr_loss(theta, A_tr, y_tr, LAM_LR))
# results['sgd'] = dict(theta=theta, time=..., history=history_sgd)

raise NotImplementedError("Implement SGD here")


In [ ]:
# TODO: your-choice fourth method (Nesterov, Adam, Newton-CG, ADMM, ...).
# Document why you chose it. Use a callback to record history.

raise NotImplementedError("Implement your fourth method here")


In [ ]:
print(f'{"method":12} {"loss":>10} {"|grad|":>10} {"time":>9} {"train acc":>10} {"test acc":>10}')
for name, info in results.items():
    th = info['theta']
    f, g = lr_loss_and_grad(th, A_tr, y_tr, LAM_LR)
    print(f'{name:12} {f:>10.6f} {np.linalg.norm(g):>10.2e} {info["time"]:>9.3f} '
          f'{lr_acc(th, A_tr, y_tr):>10.4f} {lr_acc(th, A_te, y_te):>10.4f}')

plot_loss_curves(
    [results['lbfgs']['history'], results['sgd']['history'],
     results.get('nesterov', {}).get('history', [])],
    ['L-BFGS', 'SGD', 'fourth method'],
    title='WDBC LR: training loss vs iteration')
plt.show()


### 2.4 Scaling essay (you write — markdown)

*Suppose the dataset grows to $N = 10^7$, $p = 10^5$. Which of your four methods do you keep? What additional engineering becomes necessary? What about distributed data? Adding $\ell_1$? Write 2–3 paragraphs.*

*your discussion here*


---

## 3. Problem 3 — IEEE 9-bus AC OPF [20 pts]

Standard MATPOWER `case9` benchmark. AC OPF in polar form with quadratic generation cost.

### 3.1 Load network (filled in via `load_case9`)


In [ ]:
net = load_case9()
print(f'nb = {net["nb"]} buses')
print(f'Total load = {net["P_d"].sum() * 100:.0f} MW')
print(f'Sum P_g_max = {net["P_g_max"].sum() * 100:.0f} MW')


### 3.2 Nonconvexity discussion (you write — markdown)

*1 short paragraph: identify the specific terms in the power-balance equations that fail convexity. Why is the feasible set a curved manifold?*

*your discussion here*


### 3.3 Pyomo model (you write)

*Build the Pyomo model for AC OPF. Variables for $|V|$, $\theta$, $P_g$, $Q_g$ with the appropriate bounds; constraints for the slack-bus angle, P-balance, Q-balance; quadratic objective on generator real powers. Pass to IPOPT.*


In [ ]:
# TODO: Pyomo + IPOPT model.
# from pyomo.environ import (ConcreteModel, Var, Objective, Constraint,
#                             sin, cos, SolverFactory, minimize as pyo_min)
#
# def build_pyomo_acopf(net, alpha=1.0, x0=None):
#     m = ConcreteModel()
#     m.B = range(net['nb'])
#     m.V  = Var(m.B, ..., bounds=...)
#     m.th = Var(m.B, ..., bounds=...)
#     m.Pg = Var(m.B, ..., bounds=...)
#     m.Qg = Var(m.B, ..., bounds=...)
#     m.slack = Constraint(expr=m.th[0] == 0.0)
#     # P-balance and Q-balance constraints
#     # Objective: sum c2*Pg^2 + c1*Pg + c0
#     return m
#
# m = build_pyomo_acopf(net)
# SolverFactory('ipopt').solve(m, tee=False)

raise NotImplementedError("Implement Pyomo model here (or use SLSQP fallback below)")


### 3.4 Baseline solve + multi-start + stress test (uses `solve_acopf_slsqp`, `random_start`)


In [ ]:
# TODO: baseline solve from flat start, then multi-start (30 random),
# then stress test alpha in {0.5, 1.0, 1.5, 2.0, 2.5}.
# Use solve_acopf_slsqp and random_start from §0.

raise NotImplementedError("Run baseline + multi-start + stress-test here")


### 3.5 Discussion (you write — markdown)

*1 paragraph each: (a) interpretation of the multi-start results — does case9 have one basin or many? (b) physical interpretation of the stress-test infeasibility at large $\alpha$.*

*your discussion here*


---

## 4. Problem 4 — Saddle-point dynamics: GDA vs Extragradient [20 pts]

Consider the convex-concave saddle problem
$$\min_x\ \max_y\ f(x, y) = \tfrac{\mu}{2}\|x\|^2 + x^\top B y - \tfrac{\mu}{2}\|y\|^2,$$
with $B \in \mathbb{R}^{2 \times 2}$ given. Saddle at $(x^\star, y^\star) = (0, 0)$.

**Two algorithms.**
- **Gradient descent–ascent (GDA):** $x_{k+1} = x_k - \eta\, \nabla_x f(x_k, y_k);\quad y_{k+1} = y_k + \eta\, \nabla_y f(x_k, y_k).$
- **Extragradient (Korpelevich, 1976):** predictor $\tilde x_k, \tilde y_k$ via a GDA half-step, then corrector with the gradient evaluated **at the lookahead** $(\tilde x_k, \tilde y_k)$:
  $$x_{k+1} = x_k - \eta\,\nabla_x f(\tilde x_k, \tilde y_k), \qquad y_{k+1} = y_k + \eta\,\nabla_y f(\tilde x_k, \tilde y_k).$$

### 4.1 Setup (filled in)

Reference $B$, initial point, gradients, and a 2D trajectory plotter.


**Why saddle-point problems matter.** Min-max optimization is everywhere in modern computational science:

- **Lagrangian duality.** Every constrained optimization problem you've solved this semester has a Lagrangian $L(x, \lambda)$ whose stationary point is a saddle. Methods that update $x$ and $\lambda$ together (augmented Lagrangian, ADMM, primal-dual IPMs) are saddle-point methods in disguise.
- **Adversarial machine learning.** Robust classification minimizes $\max_{\|\delta\| \le \varepsilon} L(w; x + \delta, y)$ — adversary picks worst perturbation, trainer picks best weights.
- **GANs.** Generator vs.\ discriminator: $\min_G \max_D \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$.
- **Distributionally robust optimization, fairness ML, two-player zero-sum games.** All saddle-point.

**Why a 2D toy.** The simplest case — a $2 \times 2$ bilinear example — is the canonical pathology that GAN trainers (re)discovered the hard way: naive GDA *rotates* around the saddle without converging. The lesson generalizes; the 2D version is just easy to visualize and analyze. §4.7 below scales to $n = 50$ to confirm the lesson holds away from the toy.

In [ ]:
B_mat = np.array([[1.0, 0.5], [-0.3, 1.0]])
x0 = np.array([1.0, 0.5])
y0 = np.array([0.5, -0.3])
ETA = 0.1

def grad_x(x, y, mu, B):
    """Partial gradient of f w.r.t. x: mu*x + B*y."""
    return mu * x + B @ y

def grad_y(x, y, mu, B):
    """Partial gradient of f w.r.t. y: B'*x - mu*y."""
    return B.T @ x - mu * y

def plot_trajectory(ax, xs, ys, title, color='tab:blue'):
    """2D trajectory plot in the (x_1, y_1) plane."""
    ax.plot(xs[:, 0], ys[:, 0], '.-', color=color, alpha=0.5, markersize=3, linewidth=1)
    ax.plot(xs[0, 0], ys[0, 0], 'o', color='black', markersize=8, label='start')
    ax.plot(0, 0, '*', color='red', markersize=14, label='saddle')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$y_1$')
    ax.axhline(0, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_aspect('equal', adjustable='datalim')

def distance_history(xs, ys):
    return np.sqrt(np.sum(xs * xs + ys * ys, axis=1))

print('Setup loaded. Use grad_x/grad_y, plot_trajectory, distance_history below.')


### 4.2 GDA (you write)

Implement GDA. Each iteration:
1. Compute $g_x = \nabla_x f(x_k, y_k)$, $g_y = \nabla_y f(x_k, y_k)$.
2. Update $x_{k+1} = x_k - \eta g_x$, $y_{k+1} = y_k + \eta g_y$.


In [ ]:
def gda(x0, y0, mu, B, eta=0.1, n_iter=200):
    """Run GDA for n_iter steps. Returns (xs, ys) of shape (n_iter+1, 2)."""
    xs = [x0.copy()]; ys = [y0.copy()]
    x, y = x0.copy(), y0.copy()
    for _ in range(n_iter):
        # TODO: gx = grad_x(x, y, mu, B)
        # TODO: gy = grad_y(x, y, mu, B)
        # TODO: x = x - eta * gx
        # TODO: y = y + eta * gy
        raise NotImplementedError("Implement GDA update")
        xs.append(x.copy()); ys.append(y.copy())
    return np.array(xs), np.array(ys)


# Test on three regimes
# for mu in [1.0, 0.1, 0.0]:
#     xs, ys = gda(x0, y0, mu, B_mat, eta=ETA, n_iter=200)
#     d = distance_history(xs, ys)
#     print(f'mu = {mu}: dist(0) = {d[0]:.3f}, dist(200) = {d[-1]:.4e}')


### 4.3 Extragradient (you write)

Implement extragradient. Each iteration:
1. **Predictor**: $\tilde x = x_k - \eta\, \nabla_x f(x_k, y_k)$, $\tilde y = y_k + \eta\, \nabla_y f(x_k, y_k)$.
2. **Corrector**: $x_{k+1} = x_k - \eta\, \nabla_x f(\tilde x, \tilde y)$, $y_{k+1} = y_k + \eta\, \nabla_y f(\tilde x, \tilde y)$.

Note: the corrector uses the gradient at the **lookahead** $(\tilde x, \tilde y)$, not at $(x_k, y_k)$. That's the entire trick.


In [ ]:
def extragradient(x0, y0, mu, B, eta=0.1, n_iter=200):
    """Korpelevich's extragradient. Returns (xs, ys) of shape (n_iter+1, 2)."""
    xs = [x0.copy()]; ys = [y0.copy()]
    x, y = x0.copy(), y0.copy()
    for _ in range(n_iter):
        # TODO: predictor (half-step)
        #   x_tilde = x - eta * grad_x(x, y, mu, B)
        #   y_tilde = y + eta * grad_y(x, y, mu, B)
        # TODO: corrector (full step at the lookahead)
        #   x = x - eta * grad_x(x_tilde, y_tilde, mu, B)
        #   y = y + eta * grad_y(x_tilde, y_tilde, mu, B)
        raise NotImplementedError("Implement extragradient update")
        xs.append(x.copy()); ys.append(y.copy())
    return np.array(xs), np.array(ys)


# Test on three regimes
# for mu in [1.0, 0.1, 0.0]:
#     xs, ys = extragradient(x0, y0, mu, B_mat, eta=ETA, n_iter=200)
#     d = distance_history(xs, ys)
#     print(f'mu = {mu}: dist(0) = {d[0]:.3f}, dist(200) = {d[-1]:.4e}')


### 4.4 Plot trajectories on three $\mu$ regimes (you write)

Make a $2 \times 3$ grid of trajectory plots. Top row: GDA. Bottom row: extragradient. Columns: $\mu \in \{1.0, 0.1, 0.0\}$. Use `plot_trajectory` from §4.1.


In [ ]:
# TODO: 2x3 grid of trajectory plots.
#
# fig, axes = plt.subplots(2, 3, figsize=(13, 8))
# for i, mu in enumerate([1.0, 0.1, 0.0]):
#     xs_g, ys_g = gda(x0, y0, mu, B_mat, eta=ETA, n_iter=200)
#     xs_e, ys_e = extragradient(x0, y0, mu, B_mat, eta=ETA, n_iter=200)
#     plot_trajectory(axes[0, i], xs_g, ys_g, f'GDA, mu={mu}', color='tab:blue')
#     plot_trajectory(axes[1, i], xs_e, ys_e, f'Extragradient, mu={mu}', color='tab:green')
# fig.tight_layout(); plt.show()

raise NotImplementedError("Make the 2x3 trajectory plot here")


### 4.5 Step-size sweep at $\mu = 0$ (you write)

Vary $\eta \in \{0.01, 0.1, 0.5, 1.0\}$. For each $\eta$, run 200 steps of GDA and 200 steps of extragradient. Print a table of final distances.

What is the largest $\eta$ that keeps GDA bounded? What's the best $\eta$ for extragradient?


In [ ]:
# TODO: step-size sweep table at mu=0.
#
# print(f'{"eta":>6}  {"GDA dist(200)":>14}  {"EG dist(200)":>14}')
# for eta_ in [0.01, 0.1, 0.5, 1.0]:
#     xs_g, ys_g = gda(x0, y0, 0.0, B_mat, eta=eta_, n_iter=200)
#     xs_e, ys_e = extragradient(x0, y0, 0.0, B_mat, eta=eta_, n_iter=200)
#     d_g = distance_history(xs_g, ys_g)[-1]
#     d_e = distance_history(xs_e, ys_e)[-1]
#     print(f'{eta_:>6.2f}  {d_g:>14.4e}  {d_e:>14.4e}')

raise NotImplementedError("Run the step-size sweep here")


### 4.6 Theory (you write — markdown)

**(i) Why does GDA fail at $\mu = 0$?** Write the linearized GDA update on $f(x, y) = x^\top B y$ in the form $\begin{pmatrix} x_{k+1} \\ y_{k+1} \end{pmatrix} = M_{\rm GDA} \begin{pmatrix} x_k \\ y_k \end{pmatrix}$. Compute $M_{\rm GDA}$ explicitly. For $B = I$, what are the eigenvalues of $M_{\rm GDA}$? Why is $|\lambda(M_{\rm GDA})| > 1$ a problem?

*your answer here*

**(ii) Why does extragradient succeed?** Repeat the linearization for extragradient and compute $M_{\rm EG}$. For $B = I$, show that $|\lambda(M_{\rm EG})| < 1$ for $\eta \in (0, 1)$. Which term in $M_{\rm EG}$ is responsible?

*your answer here*


### 4.7 Scale-up to $n = 50$

The 2D toy made the rotation visible. To confirm the lesson isn't a low-dimensional artifact, generate a random $B \in \mathbb{R}^{50 \times 50}$ and run both methods at $\mu = 0$. You can no longer plot trajectories, but you can plot $\|(x_k, y_k)\|$ vs.\ iteration on a log scale and see the same qualitative pattern: GDA's iterate norm grows or oscillates, extragradient drives it toward zero.

In [ ]:
# TODO: scale-up at n = 50, mu = 0
#
# rng_big = np.random.default_rng(7)
# n_big = 50
# B_big = rng_big.standard_normal((n_big, n_big)) / np.sqrt(n_big)
# x0_big = rng_big.standard_normal(n_big)
# y0_big = rng_big.standard_normal(n_big)
# xs_g, ys_g = gda(x0_big, y0_big, 0.0, B_big, eta=0.1, n_iter=500)
# xs_e, ys_e = extragradient(x0_big, y0_big, 0.0, B_big, eta=0.1, n_iter=500)
# Plot distance_history(xs_g, ys_g) and distance_history(xs_e, ys_e) on a log-y axis.

raise NotImplementedError("Implement the n=50 scale-up here")

---

## 5. Problem 5 — KKT and duality for a structured QP [20 pts]

$$\min_{x \in \mathbb{R}^n}\ \tfrac{1}{2}x^\top x \quad \text{s.t.}\ Ax = b,\ x \ge 0$$

with $A \in \mathbb{R}^{m \times n}$ full row rank, Slater feasible.


### 5.1 Part (a) — Lagrangian and KKT (you write — markdown)

*Write the Lagrangian. Derive the KKT conditions.*

*your work here*


### 5.2 Part (b) — Dual problem (you write — markdown)

*Derive the dual problem explicitly. Is strong duality guaranteed?*

*your work here*


### 5.3 Part (c) — Redundant equality and IPOPT inertia correction (you write — markdown)

*your work here*


### 5.4 Part (d) — Log-barrier perturbed KKT and central path (you write — markdown)

*your work here*


---

## 6. Problem 6 — Simplex projection from KKT [20 pts]

Prove the closed form used in Problem 1.


### 6.1 Part (a) — Lagrangian (you write — markdown)

*your work here*


### 6.2 Part (b) — KKT and the closed-form expression (you write — markdown)

*Show that $w_i^\star = \max(v_i - \theta^\star, 0)$.*

*your work here*


### 6.3 Part (c) — Existence and uniqueness of $\theta^\star$ (you write — markdown)

*your work here*


### 6.4 Part (d) — Closed form via sorting (you write — markdown)

*Derive $\theta^\star = \frac{1}{\rho}(\sum_{j \le \rho} u_{(j)} - 1)$.*

*your work here*


### 6.5 Part (e) — Complexity (you write — markdown)

*State the time and space complexity. Is $O(n)$ achievable? Compare to a generic IPM.*

*your work here*


### 6.6 Numerical verification (filled in — uses `project_simplex` from §1.2)


In [ ]:
v_test = np.array([-1.0, 0.5, 2.0, 0.1, -0.3, 1.5])
try:
    w_closed = project_simplex(v_test)
    wv = cp.Variable(len(v_test))
    p_ = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(wv - v_test)),
                    [wv >= 0, cp.sum(wv) == 1])
    p_.solve(solver='CLARABEL')
    print(f'closed: {np.round(w_closed, 6)}')
    print(f'CVXPY:  {np.round(wv.value, 6)}')
    print(f'diff:   {np.linalg.norm(w_closed - wv.value):.2e}')
except NotImplementedError:
    print('Fill in project_simplex (section 1.2) first.')
